# 成员 3：Task 2 基本面择时指标与 PIT 对齐

**我的任务**：读 2 号交付的带技术分面板 + CSMAR 三张财务报表与公告日期，
做 累计→单季→TTM 转换、六维度指标、Point-in-Time 对齐、基本面择时得分（fund_score），
并与 CSMAR 官方比率库交叉验证。

**输入**：`数据/step2_panel.csv`（2 号产出）+ `数据/fs_*.csv` 等 9 个 CSMAR 文件（已贴进本文件夹）
**输出**：`输出/step3_panel.csv` + `输出/quarterly_indicators.csv`（交给 4 号）
**交接**：向 4 号讲清楚——PIT 为什么必须做（不做会 51/51 期前视）；亏损期 PE 为什么置 NaN、由 PB 兜底。

In [1]:
from __future__ import annotations
import io, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from types import SimpleNamespace

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

BASE = os.getcwd()
RAW = os.path.join(BASE, '数据')
OUT = os.path.join(BASE, '输出')
os.makedirs(OUT, exist_ok=True)
STOCK_NAME = '牧原股份'
STOCK_CODE = '002714.SZ'
TRADING_DAYS = 243


In [2]:
class FundamentalFramework(object):
    """
    CSMAR 财务报表是【年初至报告期的累计值】，所以要：
      累计值 -> 单季值（Q2 = 半年报 - 一季报，依此类推）-> TTM（最近四个单季之和）
    再用公告日期(IAR_Rept.Annodt)做 PIT 对齐：只有公告日之后的交易日才能看到这份报表。
    六个维度（对应题目 Table3）：盈利 / 成长 / 偿债 / 运营 / 现金流质量 / 估值
    """

    # CSMAR 字段名 -> 中文含义
    BS = {"A001000000": "总资产", "A001100000": "流动资产", "A001101000": "货币资金",
          "A001123000": "存货", "A001212000": "固定资产", "A001216000": "生产性生物资产",
          "A002000000": "总负债", "A002100000": "流动负债", "A002101000": "短期借款",
          "A002201000": "长期借款", "A002203000": "应付债券", "A003000000": "股东权益合计",
          "A003100000": "归母股东权益"}
    IS = {"B001100000": "营业总收入", "B001101000": "营业收入", "B001201000": "营业成本",
          "B001200000": "营业总成本", "B001209000": "销售费用", "B001210000": "管理费用",
          "B001211000": "财务费用", "B001216000": "研发费用", "B001300000": "营业利润",
          "B001000000": "利润总额", "B002100000": "所得税费用", "B002000000": "净利润",
          "B002000101": "归母净利润"}
    CF = {"C001000000": "经营现金流净额", "C001001000": "销售商品收到的现金",
          "C002000000": "投资现金流净额", "C002006000": "购建固定资产支付的现金",
          "C003000000": "筹资现金流净额", "C003006000": "分配股利偿付利息支付的现金"}

    def __init__(self, raw: dict, panel: pd.DataFrame):
        self.raw = raw
        self.px = panel
        self.log = []

    def note(self, msg):
        self.log.append(msg)
        print("  [基本面] " + msg)

    # ---- 4.1 把 CSMAR 的科目代码换成中文名，只留合并报表 ----
    def tidy_statements(self):
        out = {}
        for key, mapping, name in [("fs_combas", self.BS, "资产负债表"),
                                   ("fs_comins", self.IS, "利润表"),
                                   ("fs_comscfd", self.CF, "现金流量表")]:
            df = self.raw[key].copy()
            df["Accper"] = pd.to_datetime(df["Accper"])
            n_raw = len(df)
            # Typrep = 'A' 合并报表、'B' 母公司报表；题目要的是集团口径 -> 只留 A
            df = df[df["Typrep"].astype(str).str.upper() == "A"]
            n_A = len(df)
            # CSMAR 里混进了 01-01 这种非标准报告期，只保留 3-31/6-30/9-30/12-31
            std = df["Accper"].dt.month.isin([3, 6, 9, 12]) & df["Accper"].dt.day.isin([30, 31])
            dropped = sorted(set(df.loc[~std, "Accper"].dt.strftime("%Y-%m-%d")))
            df = df[std].copy()
            # 同一报告期若发生差错更正(IfCorrect=1)，保留最新更正后的那一版
            if "IfCorrect" in df.columns:
                df = df.sort_values(["Accper", "IfCorrect"]).drop_duplicates("Accper", keep="last")
            else:
                df = df.drop_duplicates("Accper", keep="last")
            keep = {"Accper": "报告期"}
            for code, cn in mapping.items():
                if code in df.columns:
                    keep[code] = cn
            df = df[list(keep)].rename(columns=keep).set_index("报告期").sort_index()
            for c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            out[name] = df
            self.note("%s：原始 %d 行 -> 合并报表 %d 行 -> 标准报告期 %d 个（%s ~ %s），字段 %d 个"
                      % (name, n_raw, n_A, len(df), df.index.min().date(), df.index.max().date(),
                         len(df.columns)))
            if dropped:
                self.note("  剔除非标准报告期 %d 个：%s" % (len(dropped), dropped))
        return out

    # ---- 4.2 累计值 -> 单季值 -> TTM ----
    def to_single_and_ttm(self, bs, is_, cf):
        # 利润表/现金流量表是累计流量，需要差分；资产负债表是时点存量，直接用
        def single(df):
            d = df.copy()
            q = np.asarray(d.index.quarter)
            y = np.asarray(d.index.year)
            s = pd.DataFrame(index=d.index, columns=d.columns, dtype="float64")
            for col in d.columns:
                v = d[col].to_numpy(dtype="float64")
                out = v.copy()
                for i in range(1, len(v)):
                    if q[i] == 1 or y[i] != y[i - 1]:
                        out[i] = v[i]
                    else:
                        out[i] = v[i] - v[i - 1]
                s[col] = out
            return s

        is_q = single(is_)
        cf_q = single(cf)
        # TTM = 最近四个单季之和（不足四季时按已有季度年化会失真，直接置 NaN）
        def ttm(df):
            r = df.rolling(4).sum()
            r[df.rolling(4).count() < 4] = np.nan
            return r

        is_ttm = ttm(is_q)
        cf_ttm = ttm(cf_q)
        # 存量项取「期初(去年同期末) + 期末」的平均，这是 ROE / 周转率的标准算法
        eq_avg = (bs["归母股东权益"] + bs["归母股东权益"].shift(4)) / 2.0
        ta_avg = (bs["总资产"] + bs["总资产"].shift(4)) / 2.0
        inv_avg = (bs["存货"] + bs["存货"].shift(4)) / 2.0
        return bs, is_q, is_ttm, cf_q, cf_ttm, eq_avg, ta_avg, inv_avg

    # ---- 4.3 六维度指标 ----
    def build_indicators(self):
        st = self.tidy_statements()
        bs, is_, cf = st["资产负债表"], st["利润表"], st["现金流量表"]
        (bs, is_q, is_ttm, cf_q, cf_ttm,
         eq_avg, ta_avg, inv_avg) = self.to_single_and_ttm(bs, is_, cf)

        ind = pd.DataFrame(index=bs.index)
        # --- 盈利能力 ---
        ind["ROE_TTM"] = is_ttm["归母净利润"] / eq_avg
        ind["ROA_TTM"] = is_ttm["净利润"] / ta_avg
        rev_ttm = is_ttm["营业总收入"].fillna(is_ttm["营业收入"])
        cost_ttm = is_ttm["营业成本"]
        ind["毛利率_TTM"] = (rev_ttm - cost_ttm) / rev_ttm
        ind["销售净利率_TTM"] = is_ttm["净利润"] / rev_ttm
        # --- 成长能力（单季同比，消除猪周期的季节性）---
        ind["营收单季同比"] = is_q["营业总收入"] / is_q["营业总收入"].shift(4) - 1.0
        ind["归母净利单季同比"] = np.where(
            is_q["归母净利润"].shift(4).abs() > 1e-6,
            is_q["归母净利润"] / is_q["归母净利润"].shift(4).abs() - np.sign(is_q["归母净利润"].shift(4)),
            np.nan)
        ind["归母净利TTM同比"] = is_ttm["归母净利润"] / is_ttm["归母净利润"].shift(4).abs() - 1.0
        ind["总资产同比"] = bs["总资产"] / bs["总资产"].shift(4) - 1.0
        # --- 偿债能力 ---
        ind["资产负债率"] = bs["总负债"] / bs["总资产"]
        ind["流动比率"] = bs["流动资产"] / bs["流动负债"]
        ind["有息负债率"] = (bs["短期借款"].fillna(0) + bs["长期借款"].fillna(0)
                             + bs["应付债券"].fillna(0)) / bs["总资产"]
        # --- 运营效率 ---
        ind["总资产周转率_TTM"] = rev_ttm / ta_avg
        ind["存货周转率_TTM"] = cost_ttm / inv_avg
        ind["固定资产周转率_TTM"] = rev_ttm / ((bs["固定资产"] + bs["固定资产"].shift(4)) / 2.0)
        # 牧原特色：能繁母猪（生产性生物资产）与存栏（存货）扩张节奏
        ind["生物资产占总资产"] = bs["生产性生物资产"] / bs["总资产"]
        ind["存货占总资产"] = bs["存货"] / bs["总资产"]
        # --- 现金流质量 ---
        ocf_ttm = cf_ttm["经营现金流净额"]
        ind["净现比_TTM"] = ocf_ttm / is_ttm["净利润"].replace(0, np.nan)
        ind["经营现金流营收比_TTM"] = ocf_ttm / rev_ttm
        ind["自由现金流_TTM"] = ocf_ttm - cf_ttm["购建固定资产支付的现金"]
        ind["销售收现比_TTM"] = cf_ttm["销售商品收到的现金"] / rev_ttm
        # --- 估值（要用市值，见 4.4 之后拼接日频）---
        self.ind = ind
        self.statements = st
        self.is_ttm = is_ttm
        self.bs = bs
        self.note("单季/TTM 转换完成，指标 %d 个，报告期 %d 个（%s ~ %s）"
                  % (ind.shape[1], len(ind), ind.index.min().date(), ind.index.max().date()))
        return ind

    # ---- 4.4 PIT：公告日期对齐 ----
    def build_pit(self):
        ann = self.raw["iar_rept"].copy()
        ann["Accper"] = pd.to_datetime(ann["Accper"])
        ann["Annodt"] = pd.to_datetime(ann["Annodt"])
        # Reptyp 在 CSMAR 里是数字码 1/2/3/4（一季报/半年报/三季报/年报），不是 A/B，
        # 这里不按 Reptyp 过滤，只按报告期去重；同一报告期有多条时保留最后公告的那条（更正后）
        ann = ann.sort_values(["Accper", "Annodt"]).drop_duplicates("Accper", keep="last")
        # 差错更正提示：调整前净利润 Profita 与调整后 Profitb 差得多的期数
        if {"Profita", "Profitb"}.issubset(ann.columns):
            a = pd.to_numeric(ann["Profita"], errors="coerce")
            b = pd.to_numeric(ann["Profitb"], errors="coerce")
            rel = ((a - b).abs() / a.abs().clip(lower=1e6))
            big = int((rel > 0.01).sum())
            self.note("IAR_Rept 里调整前后净利润差异 >1%% 的报告期 %d 个；本框架一律用最终公告口径，"
                      "且只在公告日之后使用（PIT）" % big)
        ann_s = ann.set_index("Accper")["Annodt"]
        self.ind["公告日"] = ann_s.reindex(self.ind.index)
        miss = int(self.ind["公告日"].isna().sum())
        self.note("公告日期匹配 %d 个报告期，缺失 %d 个（缺失的用法定最晚披露日兜底）"
                  % (len(self.ind) - miss, miss))
        # 法定最晚披露日兜底：一季报/年报 4-30、半年报 8-31、三季报 10-31
        # 注意年报对应的是「次年」4-30
        def deadline(d):
            m = d.month
            if m == 3:
                return pd.Timestamp(d.year, 4, 30)
            if m == 6:
                return pd.Timestamp(d.year, 8, 31)
            if m == 9:
                return pd.Timestamp(d.year, 10, 31)
            return pd.Timestamp(d.year + 1, 4, 30)

        fallback = pd.Series([deadline(d) for d in self.ind.index], index=self.ind.index)
        na_mask = self.ind["公告日"].isna()
        self.ind["公告日"] = self.ind["公告日"].fillna(fallback)
        self.ind["公告日来源"] = np.where(na_mask, "法定最晚日兜底", "CSMAR公告日")
        # 披露时滞（天）——用来在报告里论证 PIT 的必要性
        self.ind["披露时滞天数"] = (self.ind["公告日"] - self.ind.index).dt.days
        self.note("披露时滞：中位数 %d 天，最长 %d 天，最短 %d 天"
                  % (self.ind["披露时滞天数"].median(), self.ind["披露时滞天数"].max(),
                     self.ind["披露时滞天数"].min()))
        # 前视偏差检查：如果用「报告期次月第一天」当可用日，有多少天会提前于真实公告日
        naive = pd.Series([pd.Timestamp(d.year, d.month, 1) + pd.offsets.MonthBegin(1)
                           for d in self.ind.index], index=self.ind.index)
        early = int((naive < self.ind["公告日"]).sum())
        self.note("若偷懒用「报告期次月初」当作可用日，会有 %d 个报告期早于真实公告日 → 构成前视偏差，被题目扣分" % early)
        return self.ind

    # ---- 4.5 估值：用公告日当天的市值算 PE_TTM / PB（PIT 安全）----
    def attach_valuation(self, panel):
        mkt_cap_at_ann = panel["mkt_cap"].reindex(
            self.ind["公告日"].values).values
        mkt_cap_at_ann = pd.Series(mkt_cap_at_ann, index=self.ind.index)
        # 公告日不是交易日时，往前找最近一个交易日（不能用未来价格）
        need_fill = mkt_cap_at_ann.isna()
        if need_fill.any():
            for d in self.ind.index[need_fill]:
                loc = panel.index.searchsorted(self.ind.loc[d, "公告日"], side="right") - 1
                if loc >= 0:
                    mkt_cap_at_ann.loc[d] = panel["mkt_cap"].iloc[loc]
        net_ttm = self.is_ttm["归母净利润"].reindex(self.ind.index)
        equity = self.bs["归母股东权益"].reindex(self.ind.index)
        self.ind["总市值_公告日"] = mkt_cap_at_ann
        self.ind["PE_TTM"] = np.where(net_ttm > 0, mkt_cap_at_ann / net_ttm.replace(0, np.nan), np.nan)
        self.ind["PB"] = mkt_cap_at_ann / equity
        self.note("估值指标：PE_TTM 有效 %d 期（亏损期为 NaN，改用 PB 与 PS 兜底）；PB 有效 %d 期"
                  % (int(self.ind["PE_TTM"].notna().sum()), int(self.ind["PB"].notna().sum())))
        return self.ind

    # ---- 4.6 六维度打分：相对自身历史的 expanding 百分位，越低越好的翻转 ----
    DIMENSIONS = {
        "盈利能力": [("ROE_TTM", +1), ("毛利率_TTM", +1), ("销售净利率_TTM", +1)],
        "成长能力": [("营收单季同比", +1), ("归母净利TTM同比", +1)],
        "偿债能力": [("资产负债率", -1), ("有息负债率", -1), ("流动比率", +1)],
        "运营效率": [("总资产周转率_TTM", +1), ("存货周转率_TTM", +1)],
        "现金流质量": [("净现比_TTM", +1), ("经营现金流营收比_TTM", +1), ("销售收现比_TTM", +1)],
        "估值水平": [("PE_TTM", -1), ("PB", -1)],
    }

    def expanding_pct(self, s, direction, min_p=8):
        x = pd.Series(s, dtype="float64")
        try:
            p = x.expanding(min_periods=min_p).rank(pct=True) * 100
        except AttributeError:
            p = x.expanding(min_periods=min_p).apply(lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
        return 100 - p if direction < 0 else p

    def build_dimension_scores(self):
        for dim, items in self.DIMENSIONS.items():
            parts = []
            for col, d in items:
                if col not in self.ind.columns:
                    continue
                sc = self.expanding_pct(self.ind[col], d)
                self.ind["分位_%s" % col] = sc
                parts.append(sc)
            self.ind[dim] = pd.concat(parts, axis=1).mean(axis=1)
        dims = list(self.DIMENSIONS.keys())
        # 等权合成基本面择时得分（题目 baseline：equal-weight）
        self.ind["基本面择时得分"] = self.ind[dims].mean(axis=1)
        # 不含估值的版本，用于比较「经营质量」与「估值」谁更重要
        self.ind["基本面得分_剔除估值"] = self.ind[[d for d in dims if d != "估值水平"]].mean(axis=1)
        self.dims = dims
        self.note("六个维度打分完成，得分可用期 %d 期（%s ~ %s）"
                  % (int(self.ind["基本面择时得分"].notna().sum()),
                     self.ind.index[self.ind["基本面择时得分"].notna()].min(),
                     self.ind.index[self.ind["基本面择时得分"].notna()].max()))
        return self.ind

    # ---- 4.7 与 CSMAR 官方财务指标数据库交叉验证 ----
    def cross_check_with_csmar(self):
        rows = []
        mapping = [("ROE_TTM", "fi_t5_profitability", "F050504C", "CSMAR净资产收益率TTM(%)"),
                   ("毛利率_TTM", "fi_t5_profitability", "F053301C", "CSMAR营业毛利率TTM(%)"),
                   ("总资产周转率_TTM", "fi_t4_efficiency", "F041705C", "CSMAR总资产周转率TTM"),
                   ("存货周转率_TTM", "fi_t4_efficiency", "F040505C", "CSMAR存货周转率TTM"),
                   ("资产负债率", "fi_t1_solvency", "F011201A", "CSMAR资产负债率(%)"),
                   ("流动比率", "fi_t1_solvency", "F010101A", "CSMAR流动比率"),
                   ("净现比_TTM", "fi_t6_cashflow", "F060101C", "CSMAR净利润现金净含量TTM")]
        for mine, file, code, cname in mapping:
            ref = self.raw[file].copy()
            ref["Accper"] = pd.to_datetime(ref["Accper"])
            if code not in ref.columns:
                continue
            # FI_T* 表每个报告期同时有合并报表(A)与母公司报表(B)两行，只留合并口径并去重
            if "Typrep" in ref.columns:
                ref = ref[ref["Typrep"].astype(str).str.upper() == "A"]
            ref = ref.drop_duplicates("Accper", keep="last").set_index("Accper")[code]
            ref = ref.reindex(self.ind.index)
            ref = pd.to_numeric(ref, errors="coerce")
            x = self.ind[mine].astype("float64")
            ok0 = x.notna() & ref.notna()
            if ok0.sum() < 5:
                continue
            # 自动识别口径差异：CSMAR 有的比率用小数(0.54)、有的用百分数(54.36)
            ratio = (x[ok0] / ref[ok0]).replace([np.inf, -np.inf], np.nan).dropna()
            med = float(ratio.median()) if len(ratio) else 1.0
            scale = 100.0 if abs(med - 100) < abs(med - 1) else 1.0
            y = ref / scale
            ok = x.notna() & y.notna()
            if ok.sum() < 5:
                continue
            corr = float(np.corrcoef(x[ok], y[ok])[0, 1])
            mad = float((x[ok] - y[ok]).abs().mean())
            rows.append({"我的指标": mine, "CSMAR字段": cname, "可比期数": int(ok.sum()),
                         "口径换算": "CSMAR值/%g" % scale,
                         "相关系数": round(corr, 6), "平均绝对差": round(mad, 6),
                         "最大绝对差": round(float((x[ok] - y[ok]).abs().max()), 6),
                         "我的中位数": round(float(x[ok].median()), 4),
                         "CSMAR中位数": round(float(y[ok].median()), 4)})
        cc = pd.DataFrame(rows)
        cc.to_csv(os.path.join(OUT, "task2_cross_check_csmar_ratios.csv"),
                  index=False, encoding="utf-8-sig")
        self.cross_check = cc
        self.note("与 CSMAR 官方比率库交叉验证 %d 个指标" % len(cc))
        return cc

    # ---- 4.8 把季度得分按公告日贴到日频 ----
    def merge_to_daily(self, panel):
        q = self.ind.copy()
        q = q[q["基本面择时得分"].notna()]
        # reindex + ffill：只有公告日当天及之后才能看到这份报表（Class04B 的对齐思想）
        daily = q["基本面择时得分"].reindex(panel.index, method="ffill")
        daily_name = {}
        for dim in self.dims:
            daily_name["fund_" + dim] = q[dim].reindex(panel.index, method="ffill")
        for k, v in daily_name.items():
            panel[k] = v
        panel["fund_score"] = daily
        panel["fund_score_ex_val"] = q["基本面得分_剔除估值"].reindex(panel.index, method="ffill")
        # 报告期标签，方便回看当天用的是哪份报表
        rpt = pd.Series(q.index, index=q.index)
        panel["fund_report_period"] = rpt.reindex(panel.index, method="ffill")
        # 日频估值百分位（进阶：把 PB 的历史分位画在价格图上）
        pb_daily = panel["mkt_cap"] / self.bs["归母股东权益"].reindex(
            panel.index, method="ffill").values
        panel["PB_daily"] = pb_daily
        try:
            panel["PB_daily_分位"] = pb_daily.expanding(min_periods=250).rank(pct=True) * 100
        except AttributeError:
            panel["PB_daily_分位"] = pb_daily.expanding(min_periods=250).apply(
                lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
        eff = panel["fund_score"].notna().sum()
        self.note("日频面板中已获得基本面得分的交易日 %d 天（占 %.1f%%），首个可用日 %s"
                  % (eff, eff / len(panel) * 100,
                     panel.index[panel["fund_score"].notna()][0].date()))
        return panel

    # ---- 4.9 输出表格与图 ----
    def run(self, panel):
        self.build_indicators()
        self.build_pit()
        self.attach_valuation(panel)
        self.build_dimension_scores()
        self.cross_check_with_csmar()
        panel = self.merge_to_daily(panel)

        # 季度指标表（题目要求交付：覆盖上市以来全部报告期）
        q_out = self.ind.copy()
        q_out.index = q_out.index.strftime("%Y-%m-%d")
        q_out.to_csv(os.path.join(OUT, "task2_quarterly_indicators.csv"), encoding="utf-8-sig")

        # 描述统计：只统计有得分的季度
        sc = self.ind[self.ind["基本面择时得分"].notna()]
        stat_cols = (["ROE_TTM", "毛利率_TTM", "营收单季同比", "资产负债率", "流动比率",
                      "总资产周转率_TTM", "存货周转率_TTM", "净现比_TTM", "PE_TTM", "PB"]
                     + self.dims + ["基本面择时得分"])
        stats = sc[stat_cols].describe(percentiles=[0.2, 0.5, 0.75]).T
        stats = stats[["count", "mean", "std", "min", "20%", "50%", "75%", "max"]]
        stats.to_csv(os.path.join(OUT, "task2_summary_stats.csv"), encoding="utf-8-sig")
        self.plot(panel)
        self.stats = stats
        return panel, q_out, stats

    def plot(self, panel):
        q = self.ind[self.ind["基本面择时得分"].notna()]
        fig, axes = plt.subplots(4, 1, figsize=(13, 13), sharex=False,
                                 gridspec_kw={"height_ratios": [1.6, 1, 1, 1]})
        ann = q["公告日"].values
        ax = axes[0]
        ax.plot(panel.index, panel["close_adj"], lw=0.9, color="#222222", label="后复权价格")
        ax.set_yscale("log")
        ax.set_ylabel("后复权价格(对数轴)")
        ax2 = ax.twinx()
        ax2.plot(pd.to_datetime(ann), q["PB_daily" if "PB_daily" in q else "PB"].values,
                 lw=1.0, color="#d62728", alpha=0.85, label="PB（公告日）")
        ax2.set_ylabel("PB", color="#d62728")
        ax.set_title("%s 价格与市净率（按公告日 PIT 对齐）" % STOCK_NAME)
        ax.grid(alpha=0.25)

        for ax, cols, ttl, yl in [
            (axes[1], ["盈利能力", "成长能力", "运营效率"], "盈利 / 成长 / 运营（0-100 历史分位）", "分位"),
            (axes[2], ["偿债能力", "现金流质量", "估值水平"], "偿债 / 现金流 / 估值（0-100 历史分位）", "分位"),
        ]:
            for c in cols:
                ax.plot(pd.to_datetime(ann), q[c].values, lw=1.2, marker="o", ms=2.5, label=c)
            ax.plot(pd.to_datetime(ann), q["基本面择时得分"].values, lw=1.8, color="black",
                    ls="--", label="基本面择时得分")
            ax.axhline(50, color="gray", lw=0.6, ls=":")
            ax.set_ylim(0, 100)
            ax.set_title(ttl)
            ax.set_ylabel(yl)
            ax.legend(fontsize=8, ncol=4, loc="upper left")
            ax.grid(alpha=0.25)

        axes[3].plot(panel.index, panel["fund_score"], lw=1.0, color="#1f77b4", label="基本面择时得分(日频,ffill)")
        axes[3].plot(panel.index, panel["tech_score"], lw=0.8, color="#ff7f0e", alpha=0.8, label="技术面得分")
        axes[3].axhline(50, color="gray", lw=0.6, ls=":")
        axes[3].set_ylim(0, 100)
        axes[3].set_title("基本面得分 vs 技术面得分（日频）")
        axes[3].legend(fontsize=8, loc="lower left", ncol=2)
        axes[3].grid(alpha=0.25)
        axes[3].xaxis.set_major_locator(mdates.YearLocator())
        axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        fig.tight_layout()
        p = os.path.join(OUT, "task2_fundamental.png")
        fig.savefig(p, dpi=140, bbox_inches="tight")
        plt.close(fig)
        print("  [图] 已保存 %s" % p)


# ====================================================================================
# SECTION 5: Task3 择时有效性检验、回测、评级与预期收益
# ====================================================================================

In [3]:
# ---- 读 2 号的面板 + CSMAR 财务报表 ----
panel = pd.read_csv(os.path.join(RAW, 'step2_panel.csv'), parse_dates=['trade_date'])
panel = panel.set_index('trade_date').sort_index()

raw = {}
for k in ['fs_combas', 'fs_comins', 'fs_comscfd', 'iar_rept',
          'fi_t1_solvency', 'fi_t4_efficiency', 'fi_t5_profitability',
          'fi_t6_cashflow', 'fi_t8_growth']:
    raw[k] = pd.read_csv(os.path.join(RAW, k + '.csv'))

fund = FundamentalFramework(raw, panel)
panel, quarterly, stats = fund.run(panel)
panel.to_csv(os.path.join(OUT, 'step3_panel.csv'), encoding='utf-8-sig')
quarterly.to_csv(os.path.join(OUT, 'quarterly_indicators.csv'), encoding='utf-8-sig')
print('step3 完成：面板', panel.shape, '基本面得分列 fund_score 已加入')
print(fund.cross_check.to_string(index=False))

  [基本面] 资产负债表：原始 118 行 -> 合并报表 64 行 -> 标准报告期 51 个（2013-12-31 ~ 2026-06-30），字段 13 个
  [基本面]   剔除非标准报告期 13 个：['2013-01-01', '2014-01-01', '2015-01-01', '2016-01-01', '2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01']
  [基本面] 利润表：原始 118 行 -> 合并报表 64 行 -> 标准报告期 51 个（2013-12-31 ~ 2026-06-30），字段 13 个
  [基本面]   剔除非标准报告期 13 个：['2013-01-01', '2014-01-01', '2015-01-01', '2016-01-01', '2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01']
  [基本面] 现金流量表：原始 118 行 -> 合并报表 64 行 -> 标准报告期 51 个（2013-12-31 ~ 2026-06-30），字段 6 个
  [基本面]   剔除非标准报告期 13 个：['2013-01-01', '2014-01-01', '2015-01-01', '2016-01-01', '2017-01-01', '2018-01-01', '2019-01-01', '2020-01-01', '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01']
  [基本面] 单季/TTM 转换完成，指标 20 个，报告期 51 个（2013-12-31 ~ 2026-06-30）
  [基本面] IAR_Rept 里调整前后净利润差异 >1% 的报告期 34 个；本框架一律用最终公告口径，且只在公告日之后使用（PIT）


C:\Users\35632\AppData\Local\Temp\ipykernel_37128\708851147.py:333: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel["PB_daily_分位"] = pb_daily.expanding(min_periods=250).rank(pct=True) * 100


  [图] 已保存 C:\Users\35632\Desktop\python期中大作业\小组版\3\我的代码\输出\task2_fundamental.png
step3 完成：面板 (3057, 101) 基本面得分列 fund_score 已加入
      我的指标           CSMAR字段  可比期数     口径换算     相关系数    平均绝对差    最大绝对差  我的中位数  CSMAR中位数
   ROE_TTM CSMAR净资产收益率TTM(%)    47 CSMAR值/1 0.991602 0.022839 0.158825 0.2103    0.2024
   毛利率_TTM  CSMAR营业毛利率TTM(%)    47 CSMAR值/1 1.000000 0.000000 0.000000 0.1905    0.1905
总资产周转率_TTM    CSMAR总资产周转率TTM    47 CSMAR值/1 1.000000 0.000000 0.000000 0.5874    0.5874
 存货周转率_TTM     CSMAR存货周转率TTM    47 CSMAR值/1 1.000000 0.000000 0.000000 2.2714    2.2714
     资产负债率     CSMAR资产负债率(%)    51 CSMAR值/1 1.000000 0.000000 0.000000 0.5436    0.5436
      流动比率         CSMAR流动比率    51 CSMAR值/1 1.000000 0.000000 0.000000 0.7885    0.7885
   净现比_TTM  CSMAR净利润现金净含量TTM    39 CSMAR值/1 1.000000 0.000000 0.000000 1.2680    1.2680
